# HUẤN LUYỆN PRE-TRAINING MÔ HÌNH BASE TRÊN GOOGLE COLAB PRO
## (DeepSeek MLA Architecture + FineWeb-Edu / SmolLM)

Notebook này được thiết kế để chạy trực tiếp trên GPU **A100 (40GB/80GB)** hoặc **L4 (24GB)** của Colab Pro.
Mọi checkpoint sẽ tự động được lưu trữ vào **Google Drive** để chống mất mát khi bị ngắt kết nối.

### 1. Kiểm Tra Phần Cứng GPU & Mount Google Drive

In [1]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "Vui lòng bật GPU trong Runtime -> Change runtime type!"
print(f"GPU hiện tại: {torch.cuda.get_device_name(0)}")
print(f"Dung lượng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Mount Google Drive để lưu checkpoint an toàn
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/R&DTeam/llm_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Thư mục lưu Checkpoint: {CHECKPOINT_DIR}")

PyTorch Version: 2.11.0+cu130
GPU hiện tại: NVIDIA A100-SXM4-80GB
Dung lượng VRAM: 79.25 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục lưu Checkpoint: /content/drive/MyDrive/R&DTeam/llm_checkpoints


### 2. Cài Đặt uv & Kéo Mã Nguồn Về Colab

In [2]:
# 1. Cài đặt công cụ quản lý gói siêu tốc uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

# 2. Kéo mã nguồn trực tiếp từ GitHub repo
!git clone https://github.com/hauiotlink/llm-from-scratch.git /content/llm
%cd /content/llm

# 3. Dùng uv cài đặt dependencies vào hệ thống chỉ trong vài giây
!uv pip install --system -e .

downloading uv 0.13.7 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
fatal: destination path '/content/llm' already exists and is not an empty directory.
/content/llm
Using Python 3.13.16 environment at: /usr
Resolved 72 packages in 294ms                                        
Prepared 1 package in 405ms                                              
Uninstalled 1 package in 0.50ms
Installed 1 package in 0.77ms0 (from file:///content/llm)   
 ~ llm-from-scratch==0.1.0 (from file:///content/llm)


### 3. Cấu Hình Kiến Trúc Mô Hình & Siêu Tham Số Huấn Luyện (Training Config)

In [3]:
from configs.model_config import LLMConfig, AttentionType
from core.model import ModernDecoderOnlyLLM
from core.tokenizer import ByteLevelBPETokenizer

# Nạp Tokenizer đã huấn luyện (hoặc train trên tập ngữ liệu mới)
tokenizer = ByteLevelBPETokenizer.load("data/tokenizer_tinystories.json")
print(f"Vocab Size: {tokenizer.vocab_size}")

# Cấu hình kiến trúc 1 TỶ THAM SỐ (1B Parameters) cho Colab Pro (GPU A100)
config = LLMConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_dim=2048,          # d_model
    num_layers=18,            # 18 transformer layers
    num_heads=16,             # 16 attention heads
    intermediate_dim=6784,    # SwiGLU FFN mở rộng
    max_seq_len=2048,         # Context length 2048
    attention_type=AttentionType.MLA,
    kv_lora_rank=512,         # Nén Latent KV Cache
    qk_rope_head_dim=64,      # Decoupled RoPE Key
    head_dim=128,             # Dimension mỗi head
)

device = torch.device("cuda")
model = ModernDecoderOnlyLLM(config, device=device)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Khởi tạo mô hình 1B thành công: {total_params:,} tham số ({total_params/1e9:.3f}B)")

Vocab Size: 2048
✅ Khởi tạo mô hình 1B thành công: 1,006,455,808 tham số (1.006B)


### 4. Nạp Dữ Liệu Stream (FineWeb-Edu hoặc SmolLM) & Khởi Chạy Pre-Training

In [4]:
from datasets import load_dataset
from data.dataset import create_pretraining_dataloader
from train import train_llm

# Stream tập dữ liệu giáo dục chất lượng cao FineWeb-Edu
print("Đang kết nối luồng dữ liệu HuggingFace...")
ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

# Giải phóng bộ nhớ đệm trước khi nạp dữ liệu
import gc
gc.collect()
torch.cuda.empty_cache()

# Tạo DataLoader với micro-batch size tối ưu cho model 1B (batch=4, accum=16 -> effective batch=64)
dataloader = create_pretraining_dataloader(
    text_stream=ds,
    tokenizer=tokenizer,
    batch_size=4,             # Micro-batch size vừa vặn VRAM
    seq_len=config.max_seq_len,
)

# Khởi chạy Huấn luyện Pre-training 1B Model với Gradient Checkpointing
loss_history = train_llm(
    model=model,
    dataloader=dataloader,
    tokenizer=tokenizer,
    max_steps=5000,           # Số steps huấn luyện mục tiêu
    grad_accum_steps=16,      # Tích lũy 16 steps -> Global batch size = 4 * 16 = 64
    peak_lr=5e-4,
    min_lr=5e-5,
    warmup_steps=100,
    save_interval=100,        # Tự động lưu checkpoint vào Drive mỗi 100 steps
    eval_interval=50,         # Sinh văn bản kiểm tra chất lượng mỗi 50 steps
    checkpoint_dir=CHECKPOINT_DIR,
    resume=True,              # Tự động resume tiếp tục từ checkpoint mới nhất nếu mất mạng
    use_gradient_checkpointing=True, # Tiết kiệm 75% VRAM Activations
    device=device,
)

Đang kết nối luồng dữ liệu HuggingFace...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

⚡ Gradient Checkpointing enabled: Saving up to 75% activation VRAM!
 Resuming training from checkpoint: /content/drive/MyDrive/R&DTeam/llm_checkpoints/llm_step_001200.pt
 Successfully restored model & optimizer states at Step 1200

STARTING LLM PRE-TRAINING (STEPS 1201 -> 5000 ON cuda)
Effective Batch Size: 64
Checkpoints saved to: /content/drive/MyDrive/R&DTeam/llm_checkpoints every 100 steps
Step 1210/5000 | Loss: 2.2584 | LR: 4.45e-04 | Speed: 10,823 tok/s
Step 1220/5000 | Loss: 2.3058 | LR: 4.44e-04 | Speed: 11,003 tok/s
Step 1230/5000 | Loss: 2.2672 | LR: 4.43e-04 | Speed: 11,062 tok/s
Step 1240/5000 | Loss: 2.2363 | LR: 4.43e-04 | Speed: 11,100 tok/s


KeyboardInterrupt: 

In [6]:
import os
import re
import shutil
import torch

CHECKPOINT_DIR = "/content/drive/MyDrive/R&DTeam/llm_checkpoints"

# 1. Quét danh sách file từ tên (Cực nhanh - 0.01s)
files = [
    f for f in os.listdir(CHECKPOINT_DIR)
    if f.startswith("llm_step_") and f.endswith(".pt")
]

if not files:
    print(" Không tìm thấy file checkpoint 'llm_step_*.pt' nào (hoặc đã được dọn trước đó)!")
else:
    # Trích xuất số step từ tên file: llm_step_005000.pt -> 5000
    def get_step(fname):
        match = re.search(r"llm_step_(\d+)\.pt", fname)
        return int(match.group(1)) if match else -1

    files_sorted = sorted(files, key=get_step)
    last_fname = files_sorted[-1]
    last_fpath = os.path.join(CHECKPOINT_DIR, last_fname)
    print(f" Checkpoint mới nhất & tốt nhất là: {last_fname} (Step {get_step(last_fname)})")

    last_dest = os.path.join(CHECKPOINT_DIR, "last.pt")
    best_dest = os.path.join(CHECKPOINT_DIR, "best.pt")

    # 2. Đổi tên file cuối cùng thành best.pt (Tức thì, không tốn thời gian copy)
    print(" Đang đổi tên thành best.pt...")
    shutil.move(last_fpath, best_dest)

    # 3. Tạo symlink hoặc copy nhẹ cho last.pt
    print(" Đang tạo liên kết last.pt...")
    if os.path.exists(last_dest):
        os.remove(last_dest)
    try:
        os.symlink("best.pt", last_dest)
    except Exception:
        shutil.copyfile(best_dest, last_dest)

    # 4. Chỉ nạp ĐÚNG 1 FILE duy nhất để tách bản Inference siêu nhẹ (tiết kiệm 70% dung lượng)
    print(" Đang nạp duy nhất 1 file best.pt để tạo bản Inference gọn nhẹ...")
    ckpt = torch.load(best_dest, map_location="cpu", weights_only=False)
    inference_ckpt = {
        "step": ckpt.get("step"),
        "loss": ckpt.get("loss"),
        "model_state_dict": ckpt["model_state_dict"],
        "config": ckpt["config"],
    }
    torch.save(inference_ckpt, os.path.join(CHECKPOINT_DIR, "best_inference.pt"))
    print(" Đã xuất: best_inference.pt (chỉ chứa weights, tải về máy rất nhanh)")

    # 5. Xóa nhanh 49 file checkpoint trung gian thừa để giải phóng dung lượng Drive
    print(f"\n Bắt đầu xóa {len(files_sorted) - 1} checkpoint thừa...")
    deleted_count = 0
    for fname in files_sorted[:-1]:
        fpath = os.path.join(CHECKPOINT_DIR, fname)
        try:
            os.remove(fpath)
            deleted_count += 1
        except Exception as e:
            print(f"Không thể xóa {fname}: {e}")

    print("\n" + "=" * 55)
    print(f" HOÀN TẤT! Đã giải phóng toàn bộ Drive (xóa {deleted_count} file thừa).")
    print(" Thư mục Google Drive hiện chỉ còn:")
    print("    best.pt           (Checkpoint hoàn chỉnh ở step cao nhất)")
    print("    last.pt           (Trỏ tới checkpoint cuối để resume nếu cần)")
    print("    best_inference.pt (Bản siêu nhẹ không optimizer, dùng để chat/deploy)")
    print("=" * 55)

 Checkpoint mới nhất & tốt nhất là: llm_step_005000.pt (Step 5000)
 Đang đổi tên thành best.pt...
 Đang tạo liên kết last.pt...
 Đang nạp duy nhất 1 file best.pt để tạo bản Inference gọn nhẹ...
 Đã xuất: best_inference.pt (chỉ chứa weights, tải về máy rất nhanh)

 Bắt đầu xóa 49 checkpoint thừa...

 HOÀN TẤT! Đã giải phóng toàn bộ Drive (xóa 49 file thừa).
 Thư mục Google Drive hiện chỉ còn:
    best.pt           (Checkpoint hoàn chỉnh ở step cao nhất)
    last.pt           (Trỏ tới checkpoint cuối để resume nếu cần)
    best_inference.pt (Bản siêu nhẹ không optimizer, dùng để chat/deploy)


In [7]:
import torch
from core.tokenizer import ByteLevelBPETokenizer
from core.model import ModernDecoderOnlyLLM
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Đang chạy Inference trên: {device}")

# 1. Nạp Tokenizer
tokenizer_path = "data/tokenizer_tinystories.json"
tokenizer = ByteLevelBPETokenizer.load(tokenizer_path)
print(f" Đã nạp Tokenizer ({tokenizer.vocab_size} tokens)")

# 2. Nạp Checkpoint (Ưu tiên best_inference.pt nhẹ nhất)
CHECKPOINT_DIR = "/content/drive/MyDrive/R&DTeam/llm_checkpoints"
ckpt_path = os.path.join(CHECKPOINT_DIR, "best_inference.pt")
if not os.path.exists(ckpt_path):
    ckpt_path = os.path.join(CHECKPOINT_DIR, "best.pt")

print(f" Đang nạp trọng số từ: {ckpt_path}...")
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

# Khởi tạo model từ config đã lưu trong checkpoint
model_config = ckpt["config"]
model = ModernDecoderOnlyLLM(model_config).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

step = ckpt.get("step", 5000)
loss = ckpt.get("loss", 0.0)
print(f" Mô hình sẵn sàng! (Huấn luyện {step} steps, Loss: {loss:.4f})")

# 3. Hàm sinh văn bản với KV Cache
def generate_text(
    prompt: str,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
    top_k: int = 40,
    top_p: float = 0.9,
):
    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )
    with torch.inference_mode():
        output_ids = model.generate(
            prompt_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            use_cache=True,
        )
    return tokenizer.decode(output_ids[0].tolist())

 Đang chạy Inference trên: cuda
 Đã nạp Tokenizer (2048 tokens)
 Đang nạp trọng số từ: /content/drive/MyDrive/R&DTeam/llm_checkpoints/best_inference.pt...
 Mô hình sẵn sàng! (Huấn luyện 5000 steps, Loss: 1.8656)


In [9]:
test_prompts = [
    "Artificial intelligence is a branch of computer science that",
    "The solar system consists of the Sun and",
    "Python is a popular programming language because",
    "Once upon a time, a young boy named Jack found",
]

for p in test_prompts:
    print(f"\n Prompt: {p}")
    print(f" Viết tiếp:\n{generate_text(p, max_new_tokens=60, temperature=0.7)}")
    print("-" * 50)


 Prompt: Artificial intelligence is a branch of computer science that
 Viết tiếp:
Artificial intelligence is a branch of computer science that deals with the concepts of intelligence, and that is based on the design and implementation of a calculation system. Intelligence is the study of the behaviors and res
--------------------------------------------------

 Prompt: The solar system consists of the Sun and
 Viết tiếp:
The solar system consists of the Sun and the Sun.
Because the sun is a solar eclipse, you can't get a full solar eclipse. The solar system is made up of a group of main solar syst
--------------------------------------------------

 Prompt: Python is a popular programming language because
 Viết tiếp:
Python is a popular programming language because it can be used for both languages and applications. It is easy to use and does not need to be updated. It can be used for any other language like Google.
How to use
F
--------------------------------------------------

 Pro